# Sentiment Analysis using NLP Pipeline & ML Models
## Internship Task 2

### Steps:
### 1. Data Loading
### 2. NLP Preprocessing
### 3. Feature Engineering
### 4. Model Training
### 5. Evaluation
### 6. Comparison

In [1]:
import pandas as pd
import numpy as np
import re
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [3]:
df = pd.read_csv("IMDB Dataset.csv")

df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)
print("\nClass Distribution:\n", df['sentiment'].value_counts())

Shape: (50000, 2)

Columns: Index(['review', 'sentiment'], dtype='object')

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [30]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to /root/nltk_data...


In [33]:
import re
import nltk
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    if not isinstance(text, str):
        return ""

    # Lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize
    tokens = text.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    # Remove short tokens
    tokens = [word for word in tokens if len(word) > 2]

    return " ".join(tokens)

In [35]:
processed = preprocess_text(sample)
print(processed[:200])

one reviewer mentioned watching episode hooked right exactly happened first thing struck brutality unflinching scene violence set right word trust show faint hearted timid show pull punch regard drug 


In [19]:
from tqdm import tqdm

tqdm.pandas()

df['clean_review'] = df['review'].progress_apply(preprocess_text)

100%|██████████| 50000/50000 [00:12<00:00, 4117.45it/s]


In [20]:
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one reviewers mentioned watching episode hooke...
1,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...
2,I thought this was a wonderful way to spend ti...,thought wonderful way spend time hot summer we...
3,Basically there's a family where a little boy ...,basically family little boy jake thinks zombie...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df['clean_review'])

y = df['sentiment']

In [38]:
from sklearn.feature_extraction.text import CountVectorizer

bow = CountVectorizer(max_features=5000)

X_bow = bow.fit_transform(df['clean_review'])

In [39]:
print("BoW shape:", X_bow.shape)
print("Sample words:", list(bow.get_feature_names_out())[:10])

BoW shape: (50000, 5000)
Sample words: ['aaron', 'abandoned', 'abc', 'abilities', 'ability', 'able', 'absence', 'absent', 'absolute', 'absolutely']


In [40]:
print("Feature matrix shape:", X.shape)
print("Sample features:", list(tfidf.get_feature_names_out())[:10])

Feature matrix shape: (50000, 5000)
Sample features: ['aaron', 'abandoned', 'abc', 'abilities', 'ability', 'able', 'absence', 'absent', 'absolute', 'absolutely']


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [41]:
# BoW train-test split
X_train_bow, X_test_bow, y_train_bow, y_test_bow = train_test_split(
    X_bow, y, test_size=0.2, random_state=42
)

In [42]:
print(X_train.shape, X_test.shape)

(40000, 5000) (10000, 5000)


In [25]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [26]:
y_pred = model.predict(X_test)

In [27]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nReport:\n", classification_report(y_test, y_pred))

Accuracy: 0.8922

Report:
               precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000



In [28]:
def predict_sentiment(text):
    cleaned = preprocess_text(text)
    vector = tfidf.transform([cleaned])
    prediction = model.predict(vector)[0]
    return prediction

In [29]:
print(predict_sentiment("This movie was amazing, I loved it!"))
print(predict_sentiment("Worst movie ever, waste of time"))

positive
negative


In [43]:
# NAIVE BAYES MODEL
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

In [44]:
# DECISION TREE MODEL
from sklearn.tree import DecisionTreeClassifier

dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

y_pred_dt = dt_model.predict(X_test)

In [45]:
# MODEL COMPARISON
from sklearn.metrics import accuracy_score

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))

Naive Bayes Accuracy: 0.8552
Decision Tree Accuracy: 0.7179


In [46]:
from sklearn.metrics import classification_report

print("===== Logistic Regression =====")
print(classification_report(y_test, y_pred))

print("===== Naive Bayes =====")
print(classification_report(y_test, y_pred_nb))

print("===== Decision Tree =====")
print(classification_report(y_test, y_pred_dt))

===== Logistic Regression =====
              precision    recall  f1-score   support

    negative       0.90      0.88      0.89      4961
    positive       0.88      0.91      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

===== Naive Bayes =====
              precision    recall  f1-score   support

    negative       0.86      0.85      0.85      4961
    positive       0.85      0.86      0.86      5039

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000

===== Decision Tree =====
              precision    recall  f1-score   support

    negative       0.71      0.73      0.72      4961
    positive       0.72      0.71      0.72      5039

    accuracy                           0.72     10000
   macro avg       0.72      0.72      0.72   

## 🔍 Model Comparison & Insights

### 1. Model Performance Comparison
- Logistic Regression achieved the highest accuracy (~89%) with balanced precision and recall.
- Naive Bayes performed slightly lower (~86%) but is computationally efficient.
- Decision Tree showed the lowest performance (~72%), likely due to overfitting on training data.

### 2. Feature Engineering Comparison
- TF-IDF performed well as it considers the importance of words rather than just frequency.
- Bag of Words is simpler but may include less informative words, affecting model performance.

### 3. Best Model
- Logistic Regression is the best model for this task due to its high accuracy and balanced performance across metrics.

### 4. Trade-offs
- Naive Bayes is faster but slightly less accurate.
- Logistic Regression is more accurate but computationally heavier.
- Decision Trees are easy to interpret but prone to overfitting.

### 5. Conclusion
- Proper preprocessing and TF-IDF vectorization significantly improve model performance.
- Logistic Regression with TF-IDF is the most reliable approach for sentiment classification in this dataset.

Note: The dataset contains only binary sentiment (positive/negative), so the model performs binary classification instead of multi-class.

## Note on Dataset

The dataset used in this project contains only two sentiment classes: **Positive** and **Negative**.
Although the task description mentions Neutral sentiment, it is not present in the dataset.
Therefore, this project implements a **binary sentiment classification model**.
